# 02 | MODIS vs VIIRS: what is different (checklist step 3)

Team checklist (مسار العمل خطوة بخطوة): 1 download → 2 clean → **3 handle the difference (resolution and sensitivity)** →
4 aggregate monthly → 5 app. This notebook is step 3: on the CLEAN data, how do MODIS and VIIRS differ, and why the
harmonization in `01_harmonization.ipynb` grids the points and uses a factor per calendar month.

Compares MODIS (BOTH satellites, Terra + Aqua) with ONE VIIRS satellite (Suomi NPP) over full years both flew, 2012-2025.
So the ratios below are two MODIS satellites against one VIIRS satellite.
Code: `syntra/compare.py`, figures `syntra/figures.py`. Fix things there, not here.

## 1. Settings and clean data

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))  # lets the notebook import syntra/

import pandas as pd
from syntra import compare, figures
from syntra.pipeline import load_country

COUNTRY = "Sudan"
FIG, TAB = ROOT / "outputs" / "figures", ROOT / "outputs" / "tables"
figures.style()

d = load_country(COUNTRY, with_2026=False)   # archive only: full years
veg = d["veg"]
print(f"{len(veg):,} clean vegetation fires, {veg.year.min()}-{veg.year.max()}")

4,156,206 clean vegetation fires, 2000-2025


## 2. Resolution: hotspots vs burned 0.1° cells
A VIIRS pixel is about 7× smaller than a MODIS pixel, so one fire gives several VIIRS hotspots. Counting burned cells instead of points removes much of that.

In [2]:
yr = compare.yearly(veg)
yr.to_csv(TAB / f"sensor_gap_by_year_{COUNTRY}.csv")
print(f"Median VIIRS/MODIS (Terra+Aqua): hotspots x{yr.hotspot_ratio.median():.2f}, burned cells x{yr.cell_ratio.median():.2f}")
per = veg[(veg.year >= 2012) & (veg.year <= 2025)].satellite.value_counts()
print(f"Hotspots, S-NPP vs one MODIS satellite: vs Aqua x{per['N'] / per['Aqua']:.1f}, vs Terra x{per['N'] / per['Terra']:.1f}")
figures.gap_by_year(yr, COUNTRY, FIG / f"08_gap_by_year_{COUNTRY}.png")
yr

Median VIIRS/MODIS (Terra+Aqua): hotspots x4.09, burned cells x1.88


Hotspots, S-NPP vs one MODIS satellite: vs Aqua x6.8, vs Terra x10.6


,modis_hotspots,viirs_hotspots,modis_cells,viirs_cells,hotspot_ratio,cell_ratio
year,,,,,,
2012,35977,139670,6884,10936,3.88,1.59
2013,34898,141756,7706,13789,4.06,1.79
2014,32526,127987,6280,11263,3.93,1.79
2015,30735,127306,6527,12208,4.14,1.87
2016,31334,128718,6209,10966,4.11,1.77
2017,32050,127992,6400,12230,3.99,1.91
2018,31164,129146,6840,12653,4.14,1.85
2019,32955,134227,7060,13781,4.07,1.95
2020,39251,158361,8704,16304,4.03,1.87


## 3. Sensitivity changes with the season
In the wet season (Jun-Sep) fires are few and small; VIIRS sees many MODIS misses. (Few fires in those months: e.g. only a few hundred MODIS hotspots in all Augusts together, so their ratios are less certain.) In the dry season (Nov-Jan) fires are big and both see them. One factor for the whole year would be wrong, so the harmonization uses one per calendar month.

In [3]:
bm = compare.by_month(veg)
bm.to_csv(TAB / f"sensor_gap_by_month_{COUNTRY}.csv")
figures.gap_by_month(bm, COUNTRY, FIG / f"09_gap_by_month_{COUNTRY}.png")
bm

,modis_hotspots,viirs_hotspots,modis_cells,viirs_cells,hotspot_ratio,cell_ratio
month,,,,,,
1,61167,245351,16216,27063,4.01,1.67
2,41529,188037,13410,25978,4.53,1.94
3,31809,134811,11944,24265,4.24,2.03
4,17012,81629,7647,19087,4.80,2.50
5,8194,47083,4264,14884,5.75,3.49
6,2417,17546,1421,7918,7.26,5.57
7,821,7268,474,3207,8.85,6.77
8,243,2146,154,1023,8.83,6.64
9,994,6676,500,1729,6.72,3.46


In [4]:
r = compare.monthly_correlation(veg)
print(f"Monthly series, MODIS vs VIIRS S-NPP ({r['n_months']} months): r hotspots {r['r_hotspots']:.3f}, r burned cells {r['r_cells']:.3f}")

Monthly series, MODIS vs VIIRS S-NPP (168 months): r hotspots 0.994, r burned cells 0.952


## 4. Same pass: does VIIRS see what MODIS sees?
Aqua (MODIS) and Suomi NPP (VIIRS) both pass at about 13:30 local time (Sudan: median 11:36 vs 11:25 UTC), so on the same day they look at the same fires. Two corrections make it fair:
- **coverage**: only 1° boxes where both satellites detected some fire that day. Aqua's swath is narrower and leaves gaps near the equator; a fire Aqua never flew over is not a miss.
- **neighbours**: a fire in the next 0.1° cell counts as seen (MODIS pixels grow to about 2 × 4.8 km at the swath edge).

Years 2012-2021, before Aqua's orbit drift. Without the corrections the result looks much more lopsided (71 % vs 22 %), which overstates the difference.

In [5]:
sp = compare.same_pass(veg)
print(f"S-NPP saw {sp['share_of_aqua_seen_by_snpp']:.0%} of Aqua's fire cell-days; "
      f"Aqua saw {sp['share_of_snpp_seen_by_aqua']:.0%} of S-NPP's.")
print(f"Median FRP of S-NPP fires: seen by both {sp['snpp_frp_median_both']:.1f} MW, "
      f"S-NPP only {sp['snpp_frp_median_snpp_only']:.1f} MW (the extra fires are smaller).")
raw = compare.same_pass(veg, years=(2012, 2025), coverage=False, neighbours=False)
print(f"Without the corrections (exact cell, 2012-2025): {raw['share_of_aqua_seen_by_snpp']:.0%} vs {raw['share_of_snpp_seen_by_aqua']:.0%}")
sp["by_month"].to_csv(TAB / f"sensor_same_pass_by_month_{COUNTRY}.csv")
figures.same_pass(sp, COUNTRY, FIG / f"10_same_pass_{COUNTRY}.png")
sp["by_month"]

S-NPP saw 93% of Aqua's fire cell-days; Aqua saw 67% of S-NPP's.
Median FRP of S-NPP fires: seen by both 8.8 MW, S-NPP only 6.2 MW (the extra fires are smaller).


Without the corrections (exact cell, 2012-2025): 71% vs 22%


,aqua_cell_days,snpp_cell_days,share_of_aqua_seen_by_snpp,share_of_snpp_seen_by_aqua
month,,,,
1,10408,20320,0.894,0.591
2,6747,12191,0.853,0.574
3,4694,7885,0.808,0.572
4,2455,3981,0.805,0.572
5,898,1557,0.815,0.554
6,300,503,0.860,0.600
7,128,208,0.938,0.736
8,37,80,0.892,0.538
9,131,201,0.893,0.801


## 5. Fire size (FRP)
FRP is per pixel and a MODIS pixel is bigger, so compare the shape, not the raw MW: VIIRS records many fires below 5 MW that MODIS almost never does. VIIRS also detects more at night.

In [6]:
fs = compare.frp_summary(veg)
fs.to_csv(TAB / f"sensor_frp_{COUNTRY}.csv")
figures.frp_hist(veg, COUNTRY, FIG / f"11_frp_{COUNTRY}.png")
fs

,hotspots,frp_p10,frp_p25,frp_median,frp_p75,frp_p90,share_below_5MW,share_night
sensor,,,,,,,,
MODIS (Terra+Aqua),495401,8.40,12.20,20.90,40.30,76.00,0.006,0.102
VIIRS S-NPP,2048624,1.02,2.37,5.09,9.29,16.97,0.491,0.278


## 6. What this means for step 4
- Do not add raw counts: one VIIRS satellite has about 4× the hotspots of both MODIS satellites together (about 7× Aqua alone), so the record would jump in 2012.
- Grid first: as burned 0.1° cells the gap is about 2×, and steady from year to year.
- Use a factor per calendar month: the gap ranges from ×1.4 (November) to ×6.8 (July).
- VIIRS sees almost all of what MODIS sees (93 % on the same pass), MODIS about two thirds of what VIIRS sees; the extra VIIRS fires are smaller. That is why the factor is above 1, not a calibration error.